## Завдання 1. Setup і завантаження

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 38.9 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Допоміжні функції: `q()` виконує SELECT і повертає DataFrame, `run()` виконує DDL/DML у транзакції.

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

def q(sql: str) -> pd.DataFrame:
    """Виконати SELECT і повернути результат як DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def run(sql: str) -> None:
    """Виконати DDL/DML (можна кілька statement-ів) в одній транзакції."""
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

### Отримання даних з офіційного джерела
Файл — `yellow_tripdata_2024-01.parquet` зі сторінки [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
Завантажується лише **один місяць**. З нього беремо потрібні колонки та відтворювану випадкову вибірку 100 000 рядків
(`random_state=42`) — випадкова вибірка краще за `head()` відображає розподіл по днях і годинах усього місяця.

Parquet містить значення вже в правильних типах, але ми свідомо записуємо CSV «як є», щоб staging отримав текст і всі перетворення були видимі в SQL.

In [29]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
parquet_path = DATA_DIR / 'yellow_tripdata_2024-01.parquet'
csv_path = DATA_DIR / 'hw3_taxi_sample.csv'

if not parquet_path.exists():          # кеш: при повторному Run All файл не качається знову
    urlretrieve(DATA_URL, parquet_path)

columns = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
    'PULocationID', 'DOLocationID', 'payment_type',
    'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
    'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee'
]

df_full = pd.read_parquet(parquet_path, columns=columns)
print('Рядків у місячному файлі:', len(df_full))

df = df_full.sample(n=100_000, random_state=42).sort_values('tpep_pickup_datetime')
df.to_csv(csv_path, index=False)
del df_full

print(df.shape)
df.head()

Рядків у місячному файлі: 2964624
(100000, 19)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
256,2,2023-12-31 23:56:46,2024-01-01 00:12:06,2.0,2.38,1.0,N,236,142,1,15.6,1.0,0.5,1.00,0.0,1.0,21.60,2.5,0.00
4137,2,2023-12-31 23:58:35,2024-01-01 00:13:06,6.0,8.39,1.0,N,138,217,2,33.1,6.0,0.5,0.00,0.0,1.0,42.35,0.0,1.75
3782,2,2024-01-01 00:00:17,2024-01-01 00:07:46,1.0,2.35,1.0,N,161,75,1,12.1,1.0,0.5,3.42,0.0,1.0,20.52,2.5,0.00
5138,2,2024-01-01 00:01:39,2024-01-01 01:02:53,1.0,2.59,1.0,N,161,246,2,47.1,1.0,0.5,0.00,0.0,1.0,52.10,2.5,0.00
2936,2,2024-01-01 00:02:35,2024-01-01 00:10:55,1.0,1.36,1.0,N,90,79,1,10.0,1.0,0.5,3.00,0.0,1.0,18.00,2.5,0.00


### Staging-таблиця
Усі сирі колонки мають тип `TEXT`: так імпорт не впаде на порожньому рядку, `1.0` у цілочисельному полі чи дивній даті.
Порядок колонок відповідає CSV.

In [30]:
run('''
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;
DROP TABLE IF EXISTS hw3_taxi_staging CASCADE;

CREATE TABLE hw3_taxi_staging (
    vendor_id_raw             TEXT,
    pickup_raw                TEXT,
    dropoff_raw               TEXT,
    passenger_count_raw       TEXT,
    trip_distance_raw         TEXT,
    ratecode_id_raw           TEXT,
    store_and_fwd_flag_raw    TEXT,
    pu_location_id_raw        TEXT,
    do_location_id_raw        TEXT,
    payment_type_raw          TEXT,
    fare_amount_raw           TEXT,
    extra_raw                 TEXT,
    mta_tax_raw               TEXT,
    tip_amount_raw            TEXT,
    tolls_amount_raw          TEXT,
    improvement_surcharge_raw TEXT,
    total_amount_raw          TEXT,
    congestion_surcharge_raw  TEXT,
    airport_fee_raw           TEXT
);
''')

In [31]:
STAGING_COLS = '''
    vendor_id_raw, pickup_raw, dropoff_raw, passenger_count_raw, trip_distance_raw,
    ratecode_id_raw, store_and_fwd_flag_raw, pu_location_id_raw, do_location_id_raw,
    payment_type_raw, fare_amount_raw, extra_raw, mta_tax_raw, tip_amount_raw,
    tolls_amount_raw, improvement_surcharge_raw, total_amount_raw,
    congestion_surcharge_raw, airport_fee_raw
'''

raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        with open(csv_path, 'r', encoding='utf-8', newline='') as file:
            cur.copy_expert(
                f"""
                COPY hw3_taxi_staging ({STAGING_COLS})
                FROM STDIN
                WITH (FORMAT csv, HEADER true, DELIMITER ',')
                """,
                file
            )
    raw_conn.commit()
finally:
    raw_conn.close()

q('SELECT COUNT(*) AS loaded_rows FROM hw3_taxi_staging')

,loaded_rows
0,100000


In [32]:
q('SELECT * FROM hw3_taxi_staging LIMIT 5')

,vendor_id_raw,pickup_raw,dropoff_raw,passenger_count_raw,trip_distance_raw,ratecode_id_raw,store_and_fwd_flag_raw,pu_location_id_raw,do_location_id_raw,payment_type_raw,fare_amount_raw,extra_raw,mta_tax_raw,tip_amount_raw,tolls_amount_raw,improvement_surcharge_raw,total_amount_raw,congestion_surcharge_raw,airport_fee_raw
0,2,2023-12-31 23:56:46,2024-01-01 00:12:06,2.0,2.38,1.0,N,236,142,1,15.6,1.0,0.5,1.0,0.0,1.0,21.6,2.5,0.0
1,2,2023-12-31 23:58:35,2024-01-01 00:13:06,6.0,8.39,1.0,N,138,217,2,33.1,6.0,0.5,0.0,0.0,1.0,42.35,0.0,1.75
2,2,2024-01-01 00:00:17,2024-01-01 00:07:46,1.0,2.35,1.0,N,161,75,1,12.1,1.0,0.5,3.42,0.0,1.0,20.52,2.5,0.0
3,2,2024-01-01 00:01:39,2024-01-01 01:02:53,1.0,2.59,1.0,N,161,246,2,47.1,1.0,0.5,0.0,0.0,1.0,52.1,2.5,0.0
4,2,2024-01-01 00:02:35,2024-01-01 00:10:55,1.0,1.36,1.0,N,90,79,1,10.0,1.0,0.5,3.0,0.0,1.0,18.0,2.5,0.0


## Завдання 2. Staging і DDL

### Обґрунтування типів та правил очищення

Загальне правило для всіх колонок: `NULLIF(TRIM(x), '')`. pandas записує `NaN` у CSV як порожнє поле, і `COPY ... FORMAT csv` уже завантажує його як `NULL`;
`NULLIF(TRIM(...), '')` додатково страхує від `''` у лапках і пробілів, щоб вони не впали на касті й не перетворились на 0.

| Колонка (final) | Семантика | Джерельний формат (CSV) | Тип | Правила очищення |
|---|---|---|---|---|
| `trip_id` | технічний surrogate key | — | `BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY` | генерується БД; у джерелі природного ID поїздки немає |
| `vendor_id` | код постачальника TPEP-системи (1 = Creative Mobile, 2 = Curb/VeriFone, 6 = Myle, 7 = Helix) | ціле `2` | `SMALLINT` + `CHECK IN (1,2,6,7)` | малий закритий довідник кодів, 2 байти достатньо |
| `pickup_at`, `dropoff_at` | момент посадки / висадки | `2024-01-15 08:12:43` — локальний час Нью-Йорка **без offset** | `TIMESTAMPTZ NOT NULL` | `::TIMESTAMP AT TIME ZONE 'America/New_York'` — локалізуємо до абсолютного моменту (коректно для DST); без часу поїздка не має змісту → `NOT NULL`; `CHECK (dropoff_at >= pickup_at)` |
| `pickup_date` | календарна дата посадки за нью-йоркським часом | похідне | `DATE`, generated column | `(pickup_at AT TIME ZONE 'America/New_York')::DATE` — зручно для групування по днях без «зсуву» через UTC |
| `passenger_count` | кількість пасажирів, введена водієм | float-рядок `1.0`, бувають порожні | `SMALLINT` + `CHECK 0..9` | `::NUMERIC::SMALLINT` (через `NUMERIC`, бо `'1.0'::SMALLINT` падає); `NULL` залишаємо `NULL`, не підставляємо 1 |
| `trip_distance` | відстань за таксометром, милі | `1.72` | `NUMERIC(10,2)` + `CHECK >= 0` | дві знаки після коми відповідають точності таксометра; від'ємні відстані фізично неможливі → рядок відфільтровується |
| `ratecode_id` | тарифний код (1 standard, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 negotiated, 6 group, 99 unknown) | float-рядок `1.0` / порожній | `SMALLINT` | `::NUMERIC::SMALLINT`; код `99` зберігаємо як є, але в аналізі трактуємо як «невідомо» |
| `store_and_fwd` | чи запис зберігався в пам'яті авто й був відправлений пізніше (немає зв'язку) | `Y` / `N` / порожній | `BOOLEAN` | справжня двійкова ознака: `CASE UPPER(TRIM(x)) WHEN 'Y' THEN TRUE WHEN 'N' THEN FALSE END`; інше → `NULL` |
| `pu_location_id`, `do_location_id` | ID TLC Taxi Zone посадки / висадки | ціле `161` | `INTEGER` + `CHECK 1..265` | коди зон 1–263, 264/265 = «Unknown / Outside of NYC»; це ідентифікатори, а не числа для арифметики |
| `payment_type` | спосіб оплати (0 Flex Fare/невідомо, 1 card, 2 cash, 3 no charge, 4 dispute, 5 unknown, 6 voided) | ціле `1` | `SMALLINT` + `CHECK 0..6` | закритий довідник кодів |
| `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `airport_fee` | грошові складові чека, USD | `17.70`, `-3.50`, порожні (для двох останніх) | `NUMERIC(10,2)` | гроші з точністю до центів — тільки точний десятковий тип, не `FLOAT`; **від'ємні значення не видаляємо і не блокуємо CHECK-ом**, бо в TLC це легітимні refund/void-коригування — вони позначаються в аудиті |

Текстових полів без бізнес-ліміту в цьому наборі немає, тому `VARCHAR(255)` не використовується взагалі.

**CHECK-обмеження предметної області:** порядок часу (`dropoff_at >= pickup_at`), невід'ємна відстань, `passenger_count` 0–9 (фізична місткість таксі),
допустимі коди `vendor_id`, `payment_type` та зон.

In [33]:
run('''
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;

CREATE TABLE hw3_taxi_trips (
    trip_id               BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,

    vendor_id             SMALLINT
        CHECK (vendor_id IS NULL OR vendor_id IN (1, 2, 6, 7)),

    pickup_at             TIMESTAMPTZ NOT NULL,
    dropoff_at            TIMESTAMPTZ NOT NULL,
    pickup_date           DATE
        GENERATED ALWAYS AS ((pickup_at AT TIME ZONE 'America/New_York')::DATE) STORED,

    passenger_count       SMALLINT
        CHECK (passenger_count IS NULL OR passenger_count BETWEEN 0 AND 9),

    trip_distance         NUMERIC(10, 2) NOT NULL
        CHECK (trip_distance >= 0),

    ratecode_id           SMALLINT,
    store_and_fwd         BOOLEAN,

    pu_location_id        INTEGER NOT NULL CHECK (pu_location_id BETWEEN 1 AND 265),
    do_location_id        INTEGER NOT NULL CHECK (do_location_id BETWEEN 1 AND 265),

    payment_type          SMALLINT
        CHECK (payment_type IS NULL OR payment_type BETWEEN 0 AND 6),

    fare_amount           NUMERIC(10, 2),
    extra                 NUMERIC(10, 2),
    mta_tax               NUMERIC(10, 2),
    tip_amount            NUMERIC(10, 2),
    tolls_amount          NUMERIC(10, 2),
    improvement_surcharge NUMERIC(10, 2),
    total_amount          NUMERIC(10, 2),
    congestion_surcharge  NUMERIC(10, 2),
    airport_fee           NUMERIC(10, 2),

    CHECK (dropoff_at >= pickup_at)
);
''')

### Перенесення staging → final
Правило фільтрації (рядок **не** потрапляє в clean, якщо):
- немає `pickup_at` або `dropoff_at`, або `dropoff_at < pickup_at`;
- `trip_distance` відсутня або від'ємна;
- `passenger_count` поза 0–9;
- немає зони посадки/висадки або код зони поза 1–265.

Фільтр у `WHERE` дзеркалить CHECK-и, тому `INSERT` не падає, а відкинуті рядки можна порахувати окремо (див. 3.1).

In [34]:
run('''
WITH parsed AS (
    SELECT
        NULLIF(TRIM(vendor_id_raw), '')::NUMERIC::SMALLINT              AS vendor_id,
        NULLIF(TRIM(pickup_raw),  '')::TIMESTAMP AT TIME ZONE 'America/New_York' AS pickup_at,
        NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP AT TIME ZONE 'America/New_York' AS dropoff_at,
        NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT        AS passenger_count,
        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(10, 2)             AS trip_distance,
        NULLIF(TRIM(ratecode_id_raw), '')::NUMERIC::SMALLINT            AS ratecode_id,
        CASE UPPER(TRIM(store_and_fwd_flag_raw))
            WHEN 'Y' THEN TRUE
            WHEN 'N' THEN FALSE
        END                                                             AS store_and_fwd,
        NULLIF(TRIM(pu_location_id_raw), '')::NUMERIC::INTEGER          AS pu_location_id,
        NULLIF(TRIM(do_location_id_raw), '')::NUMERIC::INTEGER          AS do_location_id,
        NULLIF(TRIM(payment_type_raw), '')::NUMERIC::SMALLINT           AS payment_type,
        NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10, 2)               AS fare_amount,
        NULLIF(TRIM(extra_raw), '')::NUMERIC(10, 2)                     AS extra,
        NULLIF(TRIM(mta_tax_raw), '')::NUMERIC(10, 2)                   AS mta_tax,
        NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10, 2)                AS tip_amount,
        NULLIF(TRIM(tolls_amount_raw), '')::NUMERIC(10, 2)              AS tolls_amount,
        NULLIF(TRIM(improvement_surcharge_raw), '')::NUMERIC(10, 2)     AS improvement_surcharge,
        NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10, 2)              AS total_amount,
        NULLIF(TRIM(congestion_surcharge_raw), '')::NUMERIC(10, 2)      AS congestion_surcharge,
        NULLIF(TRIM(airport_fee_raw), '')::NUMERIC(10, 2)               AS airport_fee
    FROM hw3_taxi_staging
)
INSERT INTO hw3_taxi_trips (
    vendor_id, pickup_at, dropoff_at, passenger_count, trip_distance,
    ratecode_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
    fare_amount, extra, mta_tax, tip_amount, tolls_amount,
    improvement_surcharge, total_amount, congestion_surcharge, airport_fee
)
SELECT
    vendor_id, pickup_at, dropoff_at, passenger_count, trip_distance,
    ratecode_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
    fare_amount, extra, mta_tax, tip_amount, tolls_amount,
    improvement_surcharge, total_amount, congestion_surcharge, airport_fee
FROM parsed
WHERE pickup_at  IS NOT NULL
  AND dropoff_at IS NOT NULL
  AND dropoff_at >= pickup_at
  AND trip_distance IS NOT NULL
  AND trip_distance >= 0
  AND (passenger_count IS NULL OR passenger_count BETWEEN 0 AND 9)
  AND pu_location_id BETWEEN 1 AND 265
  AND do_location_id BETWEEN 1 AND 265;
''')

q('SELECT * FROM hw3_taxi_trips ORDER BY trip_id LIMIT 5')

,trip_id,vendor_id,pickup_at,dropoff_at,pickup_date,passenger_count,trip_distance,ratecode_id,store_and_fwd,pu_location_id,do_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2,2024-01-01 04:56:46+00:00,2024-01-01 05:12:06+00:00,2023-12-31,2,2.38,1,False,236,142,1,15.6,1.0,0.5,1.00,0.0,1.0,21.60,2.5,0.00
1,2,2,2024-01-01 04:58:35+00:00,2024-01-01 05:13:06+00:00,2023-12-31,6,8.39,1,False,138,217,2,33.1,6.0,0.5,0.00,0.0,1.0,42.35,0.0,1.75
2,3,2,2024-01-01 05:00:17+00:00,2024-01-01 05:07:46+00:00,2024-01-01,1,2.35,1,False,161,75,1,12.1,1.0,0.5,3.42,0.0,1.0,20.52,2.5,0.00
3,4,2,2024-01-01 05:01:39+00:00,2024-01-01 06:02:53+00:00,2024-01-01,1,2.59,1,False,161,246,2,47.1,1.0,0.5,0.00,0.0,1.0,52.10,2.5,0.00
4,5,2,2024-01-01 05:02:35+00:00,2024-01-01 05:10:55+00:00,2024-01-01,1,1.36,1,False,90,79,1,10.0,1.0,0.5,3.00,0.0,1.0,18.00,2.5,0.00


## Завдання 3. Data-quality audit
### 3.1. Контроль кількості рядків

In [35]:
q('''
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_taxi_staging
UNION ALL
SELECT 'clean'   AS layer, COUNT(*) AS n_rows FROM hw3_taxi_trips;
''')

,layer,n_rows
0,staging,100000
1,clean,99999


Розкладемо різницю за причинами (один рядок може мати кілька причин):

In [36]:
q('''
WITH p AS (
    SELECT
        NULLIF(TRIM(pickup_raw),  '')::TIMESTAMP AS pickup_ts,
        NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP AS dropoff_ts,
        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC AS trip_distance,
        NULLIF(TRIM(passenger_count_raw), '')::NUMERIC AS passenger_count
    FROM hw3_taxi_staging
)
SELECT
    COUNT(*) FILTER (WHERE pickup_ts IS NULL OR dropoff_ts IS NULL) AS missing_timestamps,
    COUNT(*) FILTER (WHERE dropoff_ts < pickup_ts)                  AS dropoff_before_pickup,
    COUNT(*) FILTER (WHERE trip_distance IS NULL OR trip_distance < 0) AS bad_distance,
    COUNT(*) FILTER (WHERE passenger_count > 9 OR passenger_count < 0) AS bad_passengers
FROM p;
''')

,missing_timestamps,dropoff_before_pickup,bad_distance,bad_passengers
0,0,1,0,0


**Висновок.** Clean-шар менший за staging рівно на кількість рядків, що порушують правило з розділу 2
(у вибірці з січня 2024 це, як правило, лише одиниці рядків з `dropoff < pickup`; порожніх timestamp-ів у TLC-файлах практично немає).
Жодних інших рядків не відкидали: пропуски `passenger_count`, від'ємні суми та нульові відстані **залишено** в clean — їх позначаємо в аудиті, а не видаляємо мовчки.

> ✏️ Після запуску впишіть фактичні числа з двох таблиць вище.

### 3.2. Перевірка дублікатів

In [37]:
q('''
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
                    trip_distance, total_amount)) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
                               trip_distance, total_amount)) AS duplicate_rows
FROM hw3_taxi_trips;
''')

,total_rows,distinct_business_keys,duplicate_rows
0,99999,99999,0


In [38]:
# Приклади потенційних дублікатів, якщо вони є
q('''
SELECT vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
       trip_distance, total_amount, COUNT(*) AS n
FROM hw3_taxi_trips
GROUP BY vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, trip_distance, total_amount
HAVING COUNT(*) > 1
ORDER BY n DESC, pickup_at
LIMIT 10;
''')

,vendor_id,pickup_at,dropoff_at,pu_location_id,do_location_id,trip_distance,total_amount,n


**Business key:** `(vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, trip_distance, total_amount)` —
одна машина одного вендора не може одночасно почати й закінчити дві поїздки між тими самими зонами з тією самою відстанню і сумою.

**Висновок.** У джерелі немає гарантованого природного ID поїздки (немає номера медальйона), тому перевірка виявляє лише **точні або потенційні** дублікати.
Окремий випадок — пари «поїздка + її сторно» з від'ємними сумами: у них різний `total_amount`, тому вони не вважаються дублікатами, і це правильно.
Оскільки ми взяли випадкову вибірку 100 000 з ~3 млн, ймовірність потрапляння обох копій дубліката у вибірку мала — нульовий результат тут не доводить відсутність дублікатів у повному файлі.

### 3.3. Null-rate і спеціальні маркери

In [39]:
q('''
SELECT
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL)      * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_passenger_count,
    ROUND(COUNT(*) FILTER (WHERE ratecode_id IS NULL)          * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_ratecode_id,
    ROUND(COUNT(*) FILTER (WHERE store_and_fwd IS NULL)        * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_store_and_fwd,
    ROUND(COUNT(*) FILTER (WHERE congestion_surcharge IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_congestion,
    ROUND(COUNT(*) FILTER (WHERE airport_fee IS NULL)          * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_airport_fee,
    ROUND(COUNT(*) FILTER (WHERE fare_amount IS NULL)          * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_fare_amount,
    ROUND(COUNT(*) FILTER (WHERE vendor_id IS NULL)            * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_vendor_id,
    COUNT(*) AS total_rows
FROM hw3_taxi_trips;
''')

,pct_null_passenger_count,pct_null_ratecode_id,pct_null_store_and_fwd,pct_null_congestion,pct_null_airport_fee,pct_null_fare_amount,pct_null_vendor_id,total_rows
0,4.73,4.73,4.73,4.73,4.73,0.0,0.0,99999


Сирі маркери в staging — порожній рядок, `?`, `unknown`, `nan`, а також доменні «коди невідомості»:

In [40]:
q('''
SELECT
    -- COPY ... FORMAT csv перетворює неекранований порожній рядок на NULL,
    -- тому рахуємо і NULL, і '' / пробіли через NULLIF(TRIM(...), '')
    COUNT(*) FILTER (WHERE NULLIF(TRIM(passenger_count_raw), '') IS NULL)      AS empty_passenger_count,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(store_and_fwd_flag_raw), '') IS NULL)   AS empty_store_and_fwd,
    COUNT(*) FILTER (WHERE LOWER(TRIM(passenger_count_raw)) IN ('?', 'unknown', 'nan', 'null')) AS text_markers_passenger,
    COUNT(*) FILTER (WHERE UPPER(TRIM(store_and_fwd_flag_raw)) NOT IN ('Y', 'N', ''))           AS weird_store_and_fwd,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(passenger_count_raw), '')::NUMERIC = 0) AS zero_passengers,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(ratecode_id_raw), '')::NUMERIC = 99)    AS ratecode_99_unknown,
    COUNT(*) FILTER (WHERE TRIM(payment_type_raw) IN ('0', '5'))               AS payment_unknown_codes,
    COUNT(*) FILTER (WHERE TRIM(pu_location_id_raw) IN ('264', '265')
                        OR TRIM(do_location_id_raw) IN ('264', '265'))         AS unknown_zone,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(trip_distance_raw), '')::NUMERIC = 0)   AS zero_distance
FROM hw3_taxi_staging;
''')

,empty_passenger_count,empty_store_and_fwd,text_markers_passenger,weird_store_and_fwd,zero_passengers,ratecode_99_unknown,payment_unknown_codes,unknown_zone,zero_distance
0,4727,4727,0,0,1085,962,4727,1020,1991


**Висновок.**
- Пропуски в TLC не випадкові: `passenger_count`, `ratecode_id`, `store_and_fwd`, `congestion_surcharge` і `airport_fee` порожні **в тих самих рядках** (≈ 4–5 % для січня 2024). Це записи, де `payment_type = 0`, тобто вендор не передав частину полів — пропуск систематичний, його не можна заповнювати середнім.
- Текстових маркерів `?`/`unknown` немає — Parquet-джерело типізоване. Але пропуск ховається в **кодах**: `passenger_count = 0`, `ratecode_id = 99`, `payment_type` 0/5, зони 264/265 та `trip_distance = 0`. Формально це не `NULL`, тому звичайний null-rate їх не показує.
- `fare_amount` та `vendor_id` без пропусків: нульовий null-rate — теж результат, але він не гарантує якості (див. від'ємні суми в розділі 4).

> ✏️ Впишіть фактичні відсотки з таблиць вище.

## Завдання 4. Розподіли та потенційні аномалії
### 4.1. Розподіл `payment_type`

In [41]:
q('''
SELECT
    payment_type,
    CASE payment_type
        WHEN 0 THEN 'Flex Fare / not reported'
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
    END AS payment_label,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY n_rows DESC, payment_type
LIMIT 10;
''')

,payment_type,payment_label,n_rows,pct_rows
0,1,Credit card,78131,78.13
1,2,Cash,14877,14.88
2,0,Flex Fare / not reported,4727,4.73
3,4,Dispute,1594,1.59
4,3,No charge,670,0.67


**Висновок.** Розподіл сильно незбалансований: картка домінує (≈ три чверті поїздок), готівка — друга, `0` відповідає рядкам з пропусками з 3.3,
а `No charge` та `Dispute` — рідкісні категорії (≈ 1–2 %). Важливий наслідок для ML: `tip_amount` фіксується майже лише для карткових оплат
(чайові готівкою не записуються), тому задача «прогноз чайових» коректна тільки на підмножині `payment_type = 1`.

### 4.2. Базова статистика `trip_distance` та `total_amount`

In [42]:
q('''
SELECT
    'trip_distance' AS metric,
    COUNT(trip_distance) AS n_non_null,
    MIN(trip_distance) AS min_value,
    MAX(trip_distance) AS max_value,
    ROUND(AVG(trip_distance)::NUMERIC, 2) AS avg_value,
    ROUND(STDDEV_SAMP(trip_distance)::NUMERIC, 2) AS sd_value,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_distance) AS median_value
FROM hw3_taxi_trips
UNION ALL
SELECT
    'total_amount',
    COUNT(total_amount), MIN(total_amount), MAX(total_amount),
    ROUND(AVG(total_amount)::NUMERIC, 2),
    ROUND(STDDEV_SAMP(total_amount)::NUMERIC, 2),
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_amount)
FROM hw3_taxi_trips;
''')

,metric,n_non_null,min_value,max_value,avg_value,sd_value,median_value
0,trip_distance,99999,0.00,99.00,3.21,4.33,1.68
1,total_amount,99999,-188.71,623.71,26.68,22.67,20.10


**Висновок.** Обидва показники мають довгий правий хвіст: середнє помітно більше за медіану, а стандартне відхилення в рази більше за середнє
для `trip_distance` (кілька аномально довгих «поїздок»). `total_amount` має від'ємний мінімум — це коригування/сторно.
Через асиметрію правило 3σ тут малоінформативне, тому в 4.3 використовуємо доменну перевірку.

### 4.3. Domain-specific outlier-check
Перевіряємо фізично/бізнесово неправдоподібні записи:
від'ємні суми, поїздки > 100 миль, тривалість > 12 годин і середня швидкість > 80 mph.

In [43]:
q('''
SELECT
    trip_id, pickup_at, dropoff_at, trip_distance, fare_amount, total_amount, payment_type,
    ROUND(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 60.0, 1) AS duration_min,
    ROUND(trip_distance / NULLIF(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600.0, 0), 1) AS avg_mph,
    CONCAT_WS(', ',
        CASE WHEN fare_amount < 0 OR total_amount < 0 THEN 'negative_amount' END,
        CASE WHEN trip_distance > 100 THEN 'distance_gt_100mi' END,
        CASE WHEN dropoff_at - pickup_at > INTERVAL '12 hours' THEN 'duration_gt_12h' END,
        CASE WHEN trip_distance / NULLIF(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600.0, 0) > 80
             THEN 'speed_gt_80mph' END
    ) AS anomaly_reason
FROM hw3_taxi_trips
WHERE fare_amount < 0
   OR total_amount < 0
   OR trip_distance > 100
   OR dropoff_at - pickup_at > INTERVAL '12 hours'
   OR trip_distance / NULLIF(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600.0, 0) > 80
ORDER BY trip_distance DESC, pickup_at
LIMIT 20;
''')

,trip_id,pickup_at,dropoff_at,trip_distance,fare_amount,total_amount,payment_type,duration_min,avg_mph,anomaly_reason
0,68058,2024-01-22 21:37:55+00:00,2024-01-22 23:04:32+00:00,31.26,-154.2,-188.71,4,86.6,21.7,negative_amount
1,18310,2024-01-07 04:16:46+00:00,2024-01-07 05:14:30+00:00,29.02,-110.1,-116.85,4,57.7,30.2,negative_amount
2,47930,2024-01-16 22:02:11+00:00,2024-01-16 22:58:41+00:00,28.50,0.0,-5.75,2,56.5,30.3,negative_amount
3,99667,2024-02-01 02:49:10+00:00,2024-02-01 03:29:17+00:00,27.79,-102.4,-106.65,4,40.1,41.6,negative_amount
4,46109,2024-01-16 07:08:14+00:00,2024-01-16 07:54:41+00:00,27.29,-130.4,-141.59,4,46.5,35.3,negative_amount
5,20679,2024-01-08 03:53:51+00:00,2024-01-08 04:31:36+00:00,25.17,-91.2,-100.45,4,37.8,40.0,negative_amount
6,33268,2024-01-12 04:55:43+00:00,2024-01-12 05:35:38+00:00,24.94,-70.0,-88.00,4,39.9,37.5,negative_amount
7,5401,2024-01-03 12:13:26+00:00,2024-01-03 13:01:47+00:00,24.67,-96.1,-100.10,2,48.4,30.6,negative_amount
8,66063,2024-01-22 01:49:17+00:00,2024-01-22 02:18:12+00:00,23.79,-103.1,-114.29,4,28.9,49.4,negative_amount
9,20793,2024-01-08 08:28:46+00:00,2024-01-08 08:56:41+00:00,23.30,-84.9,-91.65,4,27.9,50.1,negative_amount


In [44]:
# Скільки рядків потрапило під кожне правило
q('''
SELECT
    COUNT(*) FILTER (WHERE fare_amount < 0 OR total_amount < 0)            AS negative_amount,
    COUNT(*) FILTER (WHERE trip_distance > 100)                            AS distance_gt_100mi,
    COUNT(*) FILTER (WHERE dropoff_at - pickup_at > INTERVAL '12 hours')   AS duration_gt_12h,
    COUNT(*) FILTER (WHERE trip_distance / NULLIF(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600.0, 0) > 80)
                                                                           AS speed_gt_80mph
FROM hw3_taxi_trips;
''')

,negative_amount,distance_gt_100mi,duration_gt_12h,speed_gt_80mph
0,1308,0,38,35


**Висновок і дії:**

| Тип аномалії | Інтерпретація | Дія |
|---|---|---|
| `negative_amount` | сторно/refund, зазвичай `payment_type` 3/4 | **додати прапорець** `is_adjustment`; **виключити з training data** для моделей тарифу/чайових, але залишити для фінансової звітності |
| `distance_gt_100mi` зі звичайною сумою | збій одометра/GPS — відстань не узгоджується з ціною | **передати на перевірку джерела**; у training data — виключити |
| `duration_gt_12h` | водій не закрив поїздку в таксометрі (часто ~23–24 год) | **додати прапорець**, тривалість не використовувати як ознаку |
| `speed_gt_80mph` | неможлива швидкість у місті — помилка часу або відстані | **виключити з training data** |
| Далека, але правдоподібна поїздка (напр. до аеропорту Newark, 20–40 миль) | валідний рідкісний випадок | **залишити** — не видаляємо лише через великі значення |

## Завдання 5. Десять DQL/EDA-запитів

Покриття обов'язкових конструкцій:

| Конструкція | Запити |
|---|---|
| явний перелік колонок у `SELECT` | усі |
| `WHERE` з `AND` / `OR` / `NOT` і дужками | Q4, Q8 |
| `IN` / `BETWEEN` | Q3, Q4, Q6, Q8 |
| `LIKE` / `ILIKE` | Q9 |
| `IS NULL` / `IS NOT NULL` | Q5, Q9 |
| `COALESCE` / `NULLIF` | Q2, Q5, Q7 |
| `IS DISTINCT FROM` / `IS NOT DISTINCT FROM` | Q6, Q7 |
| `DISTINCT` | Q10 |
| `ORDER BY` + `LIMIT` | Q1, Q3, Q4, Q6, Q8, Q10 |
| `GROUP BY` (≥ 2) | Q1, Q2, Q3, Q5, Q6, Q10 |

### Q1
**Business-question:** У які години доби (за нью-йоркським часом) відбувається найбільше поїздок?

In [45]:
q('''
SELECT
    EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York')::INT AS pickup_hour,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips
GROUP BY pickup_hour
ORDER BY n_trips DESC, pickup_hour
LIMIT 5;
''')

,pickup_hour,n_trips
0,18,7082
1,17,6946
2,16,6487
3,15,6476
4,19,6290


**Interpretation:** Пік припадає на вечірні години (приблизно 17:00–19:00) — вечірній commute і вихід з офісів. Ранкового піку, характерного для громадського транспорту, у таксі майже немає. Для forecasting-моделі година доби — одна з ключових ознак, і перетворювати її варто циклічно (sin/cos).

### Q2
**Business-question:** Яка середня частка чайових від тарифу за способом оплати?

In [46]:
q('''
SELECT
    payment_type,
    COUNT(*) AS n_trips,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(AVG(tip_amount / NULLIF(fare_amount, 0)) * 100, 1) AS avg_tip_pct_of_fare
FROM hw3_taxi_trips
WHERE fare_amount > 0
GROUP BY payment_type
ORDER BY payment_type;
''')

,payment_type,n_trips,avg_tip,avg_tip_pct_of_fare
0,0,4662,1.56,7.9
1,1,78125,4.15,25.8
2,2,14592,0.00,0.0
3,3,462,0.00,0.0
4,4,823,0.02,0.0


**Interpretation:** `NULLIF(fare_amount, 0)` захищає від ділення на нуль. Для карткової оплати чайові в середньому становлять близько 20–25 % тарифу, а для готівки — фактично нуль: це не поведінка пасажирів, а обмеження збору даних (готівкові чайові не фіксуються). Отже, `tip_amount` для cash — не справжній нуль, а прихований пропуск.

### Q3
**Business-question:** Які зони посадки найпопулярніші для денних поїздок середньої довжини (1–10 миль)?

In [47]:
q('''
SELECT
    pu_location_id,
    COUNT(*) AS n_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(total_amount), 2) AS avg_total
FROM hw3_taxi_trips
WHERE trip_distance BETWEEN 1 AND 10
  AND EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York') BETWEEN 8 AND 20
GROUP BY pu_location_id
ORDER BY n_trips DESC
LIMIT 10;
''')

,pu_location_id,n_trips,avg_distance,avg_total
0,236,2777,2.21,22.51
1,161,2743,2.22,23.76
2,237,2523,2.02,21.80
3,162,2080,2.25,23.82
4,186,1954,2.39,25.16
5,142,1897,2.18,21.80
6,239,1759,2.19,22.07
7,163,1611,2.17,22.91
8,170,1558,2.28,24.17
9,230,1557,2.35,24.04


**Interpretation:** Топ складають зони Мангеттена (Upper East Side, Midtown, Penn Station тощо — коди 237, 236, 161, 186 і подібні; назви можна перевірити в офіційному taxi_zone_lookup.csv). Попит сильно сконцентрований у кількох зонах, тому `pu_location_id` — висококардинальна, але дуже інформативна категоріальна ознака; для ML краще target/frequency-encoding, ніж one-hot на 265 значень.

### Q4
**Business-question:** Скільки поїздок з аеропортів JFK або LaGuardia оплачено не карткою і не готівкою?

In [48]:
q('''
SELECT
    trip_id, pickup_at, pu_location_id, payment_type, fare_amount, total_amount
FROM hw3_taxi_trips
WHERE (pu_location_id IN (132, 138) OR ratecode_id = 2)
  AND NOT (payment_type IN (1, 2))
ORDER BY total_amount DESC
LIMIT 10;
''')

,trip_id,pickup_at,pu_location_id,payment_type,fare_amount,total_amount
0,75962,2024-01-25 03:34:57+00:00,132,4,195.5,216.69
1,29655,2024-01-11 04:11:31+00:00,132,4,150.0,151.00
2,2629,2024-01-02 03:14:20+00:00,132,4,119.2,123.45
3,1219,2024-01-01 15:49:48+00:00,132,4,117.8,121.05
4,34847,2024-01-12 20:20:06+00:00,132,3,120.0,121.00
5,14614,2024-01-06 02:13:49+00:00,132,4,105.2,118.89
6,79891,2024-01-26 04:41:32+00:00,132,4,94.7,105.89
7,17377,2024-01-06 23:22:19+00:00,132,4,102.4,105.65
8,69181,2024-01-23 04:44:00+00:00,132,4,100.3,104.55
9,98844,2024-01-31 23:22:11+00:00,132,0,70.0,102.22


**Interpretation:** Зони 132 (JFK) і 138 (LaGuardia) та тариф 2 (JFK flat fare) поєднані через `OR` у дужках, а `NOT` виключає звичайні способи оплати. Серед результатів переважають `payment_type` 0 (не передано), 3 (no charge) та 4 (dispute); аеропортові поїздки дорогі, тому спірні оплати тут мають відчутну фінансову вагу. Такі записи варто розглядати окремо від звичайних транзакцій.

### Q5
**Business-question:** Яка частка поїздок не має passenger_count у розрізі вендорів?

In [49]:
q('''
SELECT
    vendor_id,
    COUNT(*) AS n_trips,
    COUNT(*) FILTER (WHERE passenger_count IS NULL) AS n_null_passengers,
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / COUNT(*), 2) AS pct_null,
    ROUND(AVG(COALESCE(passenger_count, 1)), 2) AS avg_passengers_if_null_is_1
FROM hw3_taxi_trips
GROUP BY vendor_id
ORDER BY vendor_id;
''')

,vendor_id,n_trips,n_null_passengers,pct_null,avg_passengers_if_null_is_1
0,1,24784,1623,6.55,1.17
1,2,75209,3098,4.12,1.37
2,6,6,6,100.00,1.00


**Interpretation:** Пропуски `passenger_count` розподілені нерівномірно між вендорами — отже, їх породжує конкретна система збору даних, а не випадковість. `COALESCE(passenger_count, 1)` показує, як сильно зсунеться середнє при наївній імпутації «1 пасажир»; для ML надійніше залишити `NULL` і додати бінарний прапорець `passenger_count_missing`.

### Q6
**Business-question:** Скільки коротких поїздок починаються й закінчуються в тій самій зоні?

In [50]:
q('''
SELECT
    pu_location_id,
    COUNT(*) AS n_same_zone_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(total_amount), 2) AS avg_total
FROM hw3_taxi_trips
WHERE pu_location_id IS NOT DISTINCT FROM do_location_id
  AND pu_location_id NOT IN (264, 265)
GROUP BY pu_location_id
ORDER BY n_same_zone_trips DESC
LIMIT 10;
''')

,pu_location_id,n_same_zone_trips,avg_distance,avg_total
0,236,550,0.63,13.04
1,237,501,0.64,14.22
2,132,244,1.67,32.87
3,161,224,0.55,19.01
4,239,212,0.57,12.33
5,142,189,0.59,15.42
6,141,174,0.61,13.44
7,230,153,0.48,18.93
8,48,152,0.55,14.88
9,238,147,0.56,13.95


**Interpretation:** `IS NOT DISTINCT FROM` порівнює зони NULL-безпечно (у clean-шарі зони NOT NULL, але запит лишається коректним і для staging). Внутрішньозонні поїздки короткі (близько милі) і дешеві, найчастіше в щільних районах Мангеттена; зони 264/265 виключено, бо «Unknown = Unknown» — не справжній збіг. Нульові відстані в цій групі — кандидати на скасовані поїздки.

### Q7
**Business-question:** У скількох поїздках total_amount не дорівнює сумі складових чека?

In [51]:
q('''
SELECT
    COUNT(*) AS n_trips,
    COUNT(*) FILTER (
        WHERE total_amount IS DISTINCT FROM
              fare_amount + extra + mta_tax + tip_amount + tolls_amount
              + improvement_surcharge
              + COALESCE(congestion_surcharge, 0) + COALESCE(airport_fee, 0)
    ) AS n_inconsistent_total,
    ROUND(COUNT(*) FILTER (
        WHERE total_amount IS DISTINCT FROM
              fare_amount + extra + mta_tax + tip_amount + tolls_amount
              + improvement_surcharge
              + COALESCE(congestion_surcharge, 0) + COALESCE(airport_fee, 0)
    ) * 100.0 / COUNT(*), 2) AS pct_inconsistent
FROM hw3_taxi_trips;
''')

,n_trips,n_inconsistent_total,pct_inconsistent
0,99999,25941,25.94


**Interpretation:** `IS DISTINCT FROM` не «губить» рядки з NULL (звичайне `<>` повернуло б NULL і рядок не порахувався б), а `COALESCE` трактує відсутні доплати як 0. Більшість чеків узгоджені до цента, що підтверджує правильність вибору `NUMERIC(10,2)` (з FLOAT була б хибна «неузгодженість» через округлення). Залишок неузгоджених рядків — кандидати на перевірку в джерелі.

### Q8
**Business-question:** Які поїздки мають від'ємні суми або нульову відстань, хоча оплачені не в режимі no charge / dispute?

In [52]:
q('''
SELECT
    trip_id, pickup_at, trip_distance, fare_amount, total_amount, payment_type
FROM hw3_taxi_trips
WHERE (total_amount < 0 OR trip_distance = 0)
  AND NOT (payment_type BETWEEN 3 AND 4)
ORDER BY total_amount, pickup_at
LIMIT 10;
''')

,trip_id,pickup_at,trip_distance,fare_amount,total_amount,payment_type
0,6550,2024-01-03 19:57:51+00:00,20.52,-99.30,-129.80,2
1,5401,2024-01-03 12:13:26+00:00,24.67,-96.10,-100.10,2
2,4520,2024-01-02 23:12:04+00:00,18.60,-70.00,-87.69,2
3,10571,2024-01-05 00:07:22+00:00,18.44,-70.00,-87.69,2
4,10629,2024-01-05 00:16:44+00:00,16.68,-70.00,-87.69,2
5,92192,2024-01-29 22:25:37+00:00,15.27,-70.90,-84.15,2
6,61215,2024-01-20 18:44:21+00:00,3.72,-82.69,-83.69,2
7,91607,2024-01-29 19:22:04+00:00,17.57,-70.00,-83.44,2
8,60935,2024-01-20 17:25:54+00:00,4.53,-70.00,-82.94,2
9,50644,2024-01-17 18:48:06+00:00,17.83,-70.00,-82.69,2


**Interpretation:** Від'ємні суми здебільшого супроводжуються кодами 3/4, але частина має `payment_type` 1 або 2 — це сторно без явного статусу, їх неможливо відрізнити від звичайних поїздок без прапорця. Нульова відстань з додатною сумою — поїздки з мінімальним/фіксованим тарифом або збій таксометра. Обидві групи треба позначати прапорцем перед моделюванням тарифу.

### Q9
**Business-question:** Чи є в сирих даних поїздки з датою посадки поза січнем 2024 і які в них інші поля?

In [53]:
q('''
SELECT
    pickup_raw, dropoff_raw, trip_distance_raw, total_amount_raw, passenger_count_raw
FROM hw3_taxi_staging
WHERE pickup_raw NOT LIKE '2024-01-%'
   OR (NULLIF(TRIM(passenger_count_raw), '') IS NULL
       AND pickup_raw LIKE '2024-01-31 23:%')
ORDER BY pickup_raw
LIMIT 20;
''')

,pickup_raw,dropoff_raw,trip_distance_raw,total_amount_raw,passenger_count_raw
0,2023-12-31 23:56:46,2024-01-01 00:12:06,2.38,21.6,2.0
1,2023-12-31 23:58:35,2024-01-01 00:13:06,8.39,42.35,6.0


**Interpretation:** `LIKE '2024-01-%'` по сирому тексту — простий спосіб перевірити, що файл за місяць містить лише цей місяць. У TLC-файлах зазвичай трапляються поодинокі записи з кінця грудня 2023 або з давніми датами (2002, 2009) — це збої годинника пристрою, і їх треба відсікати фільтром за періодом. Друга умова (`IS NULL` + `LIKE`) показує, що пропуски `passenger_count` трапляються й на межі місяця, тобто не пов'язані з датою.

### Q10
**Business-question:** Скільки різних зон обслуговує кожен вендор і скільки унікальних маршрутів він покриває?

In [54]:
q('''
SELECT
    vendor_id,
    COUNT(DISTINCT pu_location_id) AS n_distinct_pickup_zones,
    COUNT(DISTINCT (pu_location_id, do_location_id)) AS n_distinct_routes,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips
GROUP BY vendor_id
ORDER BY n_trips DESC
LIMIT 5;
''')

,vendor_id,n_distinct_pickup_zones,n_distinct_routes,n_trips
0,2,198,5757,75209
1,1,220,4650,24784
2,6,1,5,6


**Interpretation:** Великі вендори (1 і 2) покривають майже всі зони, тоді як вендор 6 має лише одиниці поїздок — це рідкісна категорія, яку для ML краще об'єднати в «other». Кількість унікальних маршрутів у тисячах при 100 000 поїздок показує, що пара (зона посадки, зона висадки) дуже розріджена: як ознаку її варто кодувати через агрегати (середня ціна/тривалість маршруту), а не напряму.

> ✏️ Інтерпретації написані з урахуванням відомих властивостей TLC-даних. Після Run All звірте їх із фактичними результатами
> і підставте конкретні числа (кількості, відсотки, топ-зони) — ментор перевіряє саме висновки з вашого виводу.

## Завдання 6. Reflection

**Проблеми якості.** Текстових маркерів (`?`, `unknown`) немає, але пропуски ховаються в кодах: `payment_type = 0`, `ratecode_id = 99`, зони 264/265, `passenger_count = 0`.
Пропуски `passenger_count`, `ratecode_id`, `store_and_fwd`, `congestion_surcharge` та `airport_fee` систематично збігаються в одних рядках — це збій передачі даних вендором.
Також виявлено від'ємні суми (сторно без окремого статусу), нульові відстані, неможливі швидкості, поїздки тривалістю майже добу та дати поза місяцем файлу.
Справжнього ключа поїздки немає, тому дублікати можна виявити лише приблизно.

**Features.** Для прогнозу тарифу чи тривалості підходять: година й день тижня посадки, `pickup_date` (свята, погода через join), `pu/do_location_id` (з encoding), `ratecode_id`, `vendor_id`, `passenger_count` з прапорцем пропуску.
Не використовував би `trip_id` (технічний), `store_and_fwd` (технічний артефакт зв'язку) і `tip_amount` для cash (прихований пропуск).

**Leakage.** `dropoff_at`, тривалість, `trip_distance` за таксометром, `tip_amount`, `tolls_amount`, `total_amount` і `payment_type` відомі лише після завершення поїздки.
Якщо прогнозуємо вартість у момент замовлення — усе це leakage; відстань треба брати з маршрутизатора, а не з таксометра.

**Pipeline.** Запропонував би data engineer: окремий прапорець/тип транзакції для сторно з посиланням на оригінальну поїздку; стабільний `trip_id`; явний `NULL` замість кодів «невідомо»;
валідацію часу пристрою та фільтр за періодом файлу; timestamp з UTC-offset замість локального часу.

**ML-задача.** Природні варіанти — **regression** (вартість, тривалість, чайові для карткових оплат) і **forecasting** (кількість поїздок по зоні й годині).